# Utilities

> Utility Functions

In [ ]:
#| default_exp utils

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export

# =================================
# Standard library
# =================================
import yaml
from random import randint, random as rand, choice
from dataclasses import dataclass, field
from pathlib import Path, PurePath
from types import SimpleNamespace
from typing import Any, List, Optional, Union
import inspect
from pandas import DataFrame
from sklearn.model_selection import train_test_split

# =================================
# Imaging
# =================================
from skimage import util

# =================================
# PyTorch
# =================================
from torch import (
    Tensor as torchTensor,
    squeeze as torchsqueeze,
    max as torchmax,
    from_numpy as torch_from_numpy,
    device as torch_device,
    manual_seed, randperm,
)

from torch.cuda import is_available as is_cuda_available

# =================================
# MONAI
# =================================
from monai.data import MetaTensor
from monai.utils import set_determinism
from collections.abc import Callable, Iterable, Sequence, MutableSequence, Mapping
from monai.config import PathLike
from monai.utils.misc import ensure_tuple, ensure_tuple_rep

# =================================
# fastai
# =================================
from fastai.data.all import delegates, hasattrs, L
from fastai.vision.all import store_attr, BypassNewMeta, DisplayedTransform

# =================================
# fastcore
# =================================
from fastcore.script import risinstance

# =================================
# Multiple dispatch
# =================================
from plum import dispatch as typedispatch

In [ ]:
#| export
delegates = delegates
hasattrs = hasattrs
List = List
L = L
Any = Any
store_attr = store_attr
BypassNewMeta = BypassNewMeta
DisplayedTransform = DisplayedTransform

dataclass = dataclass
field = field

risinstance = risinstance

typedispatch = typedispatch
MetaTensor = MetaTensor
set_determinism = set_determinism

Callable = Callable
Iterable = Iterable
Sequence = Sequence
MutableSequence = MutableSequence
Mapping = Mapping
Optional = Optional
Union = Union

Path = Path
PurePath = PurePath
PathLike = PathLike

ensure_tuple = ensure_tuple
ensure_tuple_rep = ensure_tuple_rep

torchTensor = torchTensor
torch_from_numpy = torch_from_numpy
torch_device = torch_device
torchsqueeze = torchsqueeze
torchmax = torchmax

is_cuda_available = is_cuda_available 

In [ ]:
show_doc(delegates)
# show_doc(hasattrs)
# show_doc(List)
# show_doc(L)
# show_doc(Any)
# show_doc(store_attr)
# show_doc(risinstance)
# show_doc(typedispatch)
# show_doc(MetaTensor)
# show_doc(set_determinism)
# show_doc(Callable)
# show_doc(Iterable)
# show_doc(Sequence)
# show_doc(PathLike)
# show_doc(torchTensor)
# show_doc(torch_from_numpy)
# show_doc(torch_device)

---

[source](https://github.com/AnswerDotAI/fastcore/blob/main/fastcore/meta.py#LNone){target="_blank" style="float:right; font-size:smaller"}

### delegates

```python

def delegates(
    to:function=None, # Delegatee
    keep:bool=False, # Keep `kwargs` in decorated function?
    but:list=None, # Exclude these parameters from signature
    sort_args:bool=False, # Sort arguments alphabetically, doesn't work with call_parse
):


```

*Decorator: replace `**kwargs` in signature with params from `to`*

The utils module contains helper functions and classes to facilitate data manipulation, model setup, and training. These utilities add flexibility and convenience, supporting rapid experimentation and efficient data handling.


In [ ]:
#| export
def add_method(cls):
    def decorator(func):
        setattr(cls, func.__name__, func)
        return func
    return decorator

In [ ]:
#| export
def attributesFromDict(d):
    """
    The `attributesFromDict` function simplifies the conversion of dictionary keys and values into object attributes, allowing dynamic attribute creation for configuration objects. This utility is handy for initializing model or dataset configurations directly from dictionaries, improving code readability and maintainability.
    """
    self = d.pop('self')
    for n, v in d.items():
        setattr(self, n, v)

In [ ]:
#| export
def get_device():
    """
    The `get_device` function is used to detect if the device the code is executed in has got a CUDA-enabled GPU available. 
    If it doesn’t, it returns CPU. 
    """ 
    return torch_device("cuda" if is_cuda_available() else "cpu")

In [ ]:
#| export
def img2float(image, force_copy=False):
    """
    The `img2float` function turns an image into float representation.
    """
    return util.img_as_float(image, force_copy=force_copy)

In [ ]:
#| export
def img2Tensor(image):
    """
    The `img2Tensor` function turns an image into tensor representation after turning it first into float representation. 
    """
    return torchTensor(img2float(image))

In [ ]:
#| export
def route_kwargs(func, kwargs):
    """
    Filter a dictionary of kwargs to only include those accepted by `func`.

    Handles:
      - Explicit parameters
      - Functions with **kwargs (all extra keys are allowed)
    """
    sig = inspect.signature(func)
    accepts_kwargs = any(
        p.kind == inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values()
    )

    if accepts_kwargs:
        # If func accepts **kwargs, pass everything
        return kwargs.copy()
    else:
        # Otherwise, filter to matching parameters only
        return {k: v for k, v in kwargs.items() if k in sig.parameters}

Routes kwargs only to functions that accept them.

In [ ]:
#| export
def read_yaml(yaml_path):
    "Reads a YAML file and returns its contents as a dictionary"
    with open(yaml_path, 'r') as file:
        config = yaml.safe_load(file)
    return config 

In [ ]:
#| export
def read_args_from_yaml(yaml_path):
    """Reads arguments from a YAML file and converts them into a namespace."""
    config = read_yaml(yaml_path)
    if config is None:
        config = {}

    def _convert(value):
        if isinstance(value, dict):
            return SimpleNamespace(**{k: _convert(v) for k, v in value.items()})
        if isinstance(value, list):
            return [_convert(v) for v in value]
        return None if value == "None" else value

    return _convert(config)

In [ ]:
#| export
def dictlist_to_funclist(transform_dicts):
    transforms = []
    for trans in transform_dicts:
        if isinstance(trans, str):  
            transform_obj = globals().get(trans)
            transforms.append(transform_obj)
        else: 
            name, params = next(iter(trans.items()))
            transform_obj = globals().get(name) or eval(name) 
            transforms.append(transform_obj(**params))

    return transforms

In [ ]:
#| export
def dict2string(d, # The dictionary to convert.
                item_sep="_", # The separator between dictionary items (default is ", ").
                key_value_sep="", # The separator between keys and values (default is ": ").
                pad_zeroes=None, # The minimum width for integer values, padded with zeros. If None, no padding is applied.
                ):
    """
    Transforms a dictionary into a string with customizable separators and optional zero padding for integers.

    Returns the formatted dictionary as a string.
    """
    def format_value(value):
        if isinstance(value, int) and pad_zeroes is not None:
            return f"{value:0{pad_zeroes}d}"
        return str(value)
    
    return item_sep.join(f"{k}{key_value_sep}{format_value(v)}" for k, v in d.items())


In [ ]:
my_dict = {'C': 2, 'Z': 30, 'S': 1}
result = dict2string(my_dict, pad_zeroes=3)
test_eq(result, 'C002_Z030_S001')

In [ ]:
#| export 
def add_columns_to_csv(csv_path, # Path to the input CSV file
                       column_data, # Dictionary of column names and values to add. Each value can be a scalar (single value for all rows) or a list matching the number of rows.
                       output_path=None, # Path to save the updated CSV file. If None, it overwrites the input CSV file.
                       ):
    """
    Adds one or more new columns to an existing CSV file.

    """
    # Load the CSV file into a DataFrame
    df = pd.read_csv(csv_path)

    # Iterate over each column and add to the DataFrame
    for column_name, column_values in column_data.items():
        # Check if column_values is a list and matches DataFrame length
        if isinstance(column_values, list) and len(column_values) != len(df):
            raise ValueError(f"Length of values for column '{column_name}' does not match the number of rows in the CSV.")
        
        # Add the new column
        df[column_name] = column_values

    # Save the updated DataFrame to a CSV file
    output_path = output_path or csv_path
    df.to_csv(output_path, index=False)

    print(f"Columns {list(column_data.keys())} added successfully. Updated file saved to '{output_path}'")

Binary Index Splitters

In [ ]:
#| export
def _remaining_idxs(all_idxs, used_idxs):
    "Return indices not present in used indices."

    used = set(used_idxs)

    return [
        i for i in all_idxs
        if i not in used
    ]

In [ ]:
#| export
def ColSplitter(col='is_valid', on=None, **kwargs):
    "Split items (DataFrame or list[dict]) by value in `col`"

    def _inner(o, **kwargs):

        # ----------------------------
        # Normalize input
        # ----------------------------
        if isinstance(o, DataFrame):
            data = o.to_dict("records")
        elif isinstance(o, list):
            data = o
        else:
            raise TypeError("ColSplitter supports DataFrame or list[dict]")

        if not data:
            return [], []

        # ----------------------------
        # Build mask
        # ----------------------------
        if isinstance(col, int):
            values = [row[list(row.keys())[col]] for row in data]
        else:
            values = [row.get(col) for row in data]

        if on is None:
            valid_mask = [bool(v) for v in values]
        elif isinstance(on, (list, tuple, set)):
            valid_mask = [v in on for v in values]
        else:
            valid_mask = [v == on for v in values]

        # ----------------------------
        # Convert mask → indices (fastai-style)
        # ----------------------------
        train_idx = [i for i, v in enumerate(valid_mask) if not v]
        valid_idx = [i for i, v in enumerate(valid_mask) if v]

        return train_idx, valid_idx
    
    # -------------------------------------
    # Splitter metadata
    # -------------------------------------
    _inner.split_names = kwargs.get("split_names", ("train", "valid"))
    _inner.col = col
    _inner.on = on

    return _inner


def TrainTestSplitter(
    test_fraction=0.2,
    random_state=None,
    stratify=None,
    train_fraction=None,
    shuffle=True, 
    **kwargs
):
    "Split items into random train/valid subsets using sklearn train_test_split."

    def _inner(o, **kwargs):

        n = len(o)
        idxs = list(range(n))

        train_idx, valid_idx = train_test_split(
            idxs,
            test_size=test_fraction,
            random_state=random_state,
            stratify=stratify,
            train_size=train_fraction,
            shuffle=shuffle
        )

        return list(train_idx), list(valid_idx)
    
    _inner.split_names = kwargs.get("split_names", ("train", "valid"))
    _inner.test_fraction = test_fraction
    _inner.random_state = random_state
    _inner.stratify = stratify
    _inner.train_fraction = train_fraction
    _inner.shuffle = shuffle

    return _inner



def FuncSplitter(func, **kwargs):
    """
    Split items using a boolean function.

    Items where ``func(item)`` is True go to validation.
    Remaining items go to training.

    Compatible with fastai-style splitters.

    Args:
        func:
            Function returning True for validation items.

    Returns:
        Callable returning:
        ``(train_idx, valid_idx)``
    """

    def _inner(o, **kwargs):

        valid_idx = [
            i for i, x in enumerate(o)
            if func(x)
        ]

        train_idx = _remaining_idxs(
            range(len(o)),
            valid_idx,
        )

        return list(train_idx), list(valid_idx)
    
    _inner.split_names = kwargs.get("split_names", ("train", "valid"))
    _inner.func = func

    return _inner

In [ ]:
splitter = ColSplitter
print(getattr(
        splitter(),
        "split_names",
        None,
    ))

('train', 'valid')


In [ ]:
data = [
        {'id': 10, 'split': 'train'}, 
        {'id': 20, 'split': 'valid'}, 
        {'id': 30, 'split': 0},
    ]
    
splitter = ColSplitter(col='split', on='valid')
# everything that is not 'valid' should be in train, so we should get 2 in train and 1 in valid
result = splitter(data)

# Should be case-insensitive
assert len(result[0]) == 2
assert len(result[1]) == 1

Ternary splitters

In [ ]:
#| export
def NameSplitter(
    col='split_name',
    split_names=('train', 'valid'),
    **kwargs,
):
    """
    Split items by values in a column.

    Supports pandas DataFrames or ``list[dict]`` inputs and returns
    one index list per name in ``split_names``.

    By default, returns two splits for fastai compatibility:
    ``(train_idx, valid_idx)``.

    Args:
        col:
            Column name or column index containing split labels.

        split_names:
            Sequence of split names to extract indices for.

    Returns:
        Callable:
            Function returning a tuple of index lists in the
            same order as ``split_names``.
    """

    if not isinstance(split_names, (list, tuple)):
        split_names = [split_names]

    def _inner(o, **kwargs):

        # ----------------------------
        # Normalize input
        # ----------------------------
        if isinstance(o, DataFrame):
            data = o.to_dict("records")

        elif isinstance(o, list):
            data = o

        else:
            raise TypeError(
                "NameSplitter supports "
                "DataFrame or list[dict]"
            )

        if not data:
            return tuple([] for _ in split_names)

        # ----------------------------
        # Extract split column values
        # ----------------------------
        if isinstance(col, int):
            values = [
                row[list(row.keys())[col]]
                for row in data
            ]
        else:
            values = [
                row.get(col)
                for row in data
            ]

        # ----------------------------
        # Build index lists
        # ----------------------------
        idx_lists = []

        for split_name in split_names:

            if split_name not in values:
                raise ValueError(
                    f"Custom name '{split_name}' "
                    f"not found in column '{col}'"
                )

            idx = [
                i for i, v in enumerate(values)
                if v == split_name
            ]

            idx_lists.append(idx)

        return tuple(idx_lists)
    
    _inner.split_names = split_names
    _inner.col = col

    return _inner

In [ ]:
data = [
        {'id': 10, 'split': 'train'},  # Mixed case
        {'id': 20, 'split': 'valid'},  # Uppercase
        {'id': 30, 'split': 'test'},
    ]
    
splitter = NameSplitter(col='split')
result = splitter(data)
print(result)

assert len(result[0]) == 1
assert len(result[1]) == 1

splitter = NameSplitter(col='split', split_names='test') 
# with only one string, it returns all items that match that string in valid and nothing in train
result = splitter(data)

assert len(result) == 1
print(result)

splitter = NameSplitter(col='split', split_names=['train', 'valid', 'test']) 
# with only one string, it returns all items that match that string in valid and nothing in train
result = splitter(data)
print(result)

assert len(result) == 3

([0], [1])
([2],)
([0], [1], [2])


In [ ]:
#| export
def RandomSplitter(
    split_names=("train", "valid"),
    valid_fraction=0.2,
    test_fraction=0.1,
    random_state=None,
    stratify=None,
    shuffle=True,
    **kwargs,
):
    """
    Randomly split items into subsets.

    Uses ``sklearn.train_test_split``.

    The number of returned splits depends on
    ``split_names``:

    - ``("train", "valid")``
        returns ``(train_idx, valid_idx)``

    - ``("train", "valid", "test")``
        returns ``(train_idx, valid_idx, test_idx)``

    Args:
        split_names:
            Names of output splits.

            Supported values:

            - ``("train", "valid")``
            - ``("train", "valid", "test")``

        valid_fraction:
            Fraction assigned to validation.

        test_fraction:
            Fraction assigned to test.

            Ignored when only two splits
            are requested.

        random_state:
            Random seed for reproducibility.

        stratify:
            Labels used for stratified splitting.

            Can be:

            - sequence of labels
            - column name for dict-like items

        shuffle:
            Whether to shuffle before splitting.

    Returns:
        Callable returning index lists in the
        same order as ``split_names``.
    """

    split_names = tuple(split_names)

    supported = {
        ("train", "valid"),
        ("train", "valid", "test"),
    }

    if split_names not in supported:
        raise ValueError(
            "split_names must be either "
            "('train', 'valid') or "
            "('train', 'valid', 'test')"
        )

    if valid_fraction <= 0:
        raise ValueError(
            "valid_fraction must be > 0"
        )

    if test_fraction < 0:
        raise ValueError(
            "test_fraction must be >= 0"
        )

    if len(split_names) == 3:

        if valid_fraction + test_fraction >= 1:
            raise ValueError(
                "valid_fraction + test_fraction "
                "must be < 1"
            )

    def _inner(o, **kwargs):

        n = len(o)
        idxs = list(range(n))

        # ---------------------------------
        # Optional stratification labels
        # ---------------------------------
        stratify_labels = None

        if stratify is not None:

            if isinstance(stratify, str):

                stratify_labels = [
                    item[stratify]
                    for item in o
                ]

            else:
                stratify_labels = stratify

        # =================================
        # TWO-WAY SPLIT
        # =================================
        if len(split_names) == 2:

            split1_idx, split2_idx = train_test_split(
                idxs,
                test_size=valid_fraction,
                random_state=random_state,
                stratify=stratify_labels,
                shuffle=shuffle,
            )

            return (
                list(split1_idx),
                list(split2_idx)
            )

        # =================================
        # THREE-WAY SPLIT
        # =================================
        train_valid_idx, test_idx = train_test_split(
            idxs,
            test_size=test_fraction,
            random_state=random_state,
            stratify=stratify_labels,
            shuffle=shuffle,
        )

        # Stratification labels for second split
        second_stratify = None

        if stratify_labels is not None:

            second_stratify = [
                stratify_labels[i]
                for i in train_valid_idx
            ]

        # Validation size relative to remaining data
        relative_valid_size = (
            valid_fraction / (1 - test_fraction)
        )

        train_idx, valid_idx = train_test_split(
            train_valid_idx,
            test_size=relative_valid_size,
            random_state=random_state,
            stratify=second_stratify,
            shuffle=shuffle,
        )

        return (
            list(train_idx),
            list(valid_idx),
            list(test_idx),
        )
    
    _inner.split_names = split_names
    _inner.valid_fraction = valid_fraction
    _inner.test_fraction = test_fraction
    _inner.random_state = random_state
    _inner.stratify = stratify
    _inner.shuffle = shuffle

    return _inner

In [ ]:
#| export
def _get_path(o, key="image"):
    "Extract path from dict item or raw path-like object."

    if isinstance(o, dict):
        return Path(o.get(key, o.get("path", "")))

    return Path(o)


def _normalize_names(names):
    "Ensure split names are always a list."

    if not isinstance(names, (list, tuple)):
        names = [names]

    return list(names)


def _folder_idxs(
    items,
    name,
    folder="grandparent",
    key="image",
):
    """
    Return indices matching a folder name.

    Args:
        items:
            Dataset items.

        name:
            Folder name to match.

        folder:
            One of:
            ``"parent"``
            ``"grandparent"``.

        key:
            Dictionary key containing path.
    """

    idxs = []

    for i, o in enumerate(items):

        path = _get_path(o, key)

        folder_name = (
            path.parent.name
            if folder == "parent"
            else path.parent.parent.name
        )

        if folder_name == name:
            idxs.append(i)

    return idxs

In [ ]:
#| export
def GrandparentSplitter(
    split_names=("train", "valid"),
    key="image",
    **kwargs,
):
    """
    Split items by grandparent folder names.

    Example structure::

        dataset/
            train/
                cats/img1.jpg
            valid/
                dogs/img2.jpg
            test/
                dogs/img3.jpg

    Args:
        split_names:
            Grandparent folder names to split on.

        key:
            Dictionary key containing file path.

    Returns:
        Callable returning one index list
        per name in ``split_names``.
    """

    split_names = _normalize_names(split_names)

    def _inner(o, **kwargs):

        return tuple(
            _folder_idxs(
                o,
                name=name,
                folder="grandparent",
                key=key,
            )
            for name in split_names
        )
    
    _inner.split_names = split_names
    _inner.key = key

    return _inner


def ParentSplitter(
    split_names=("train", "valid"),
    key="image",
    **kwargs
):
    """
    Split items by parent folder names.

    Args:
        split_names:
            Parent folder names to split on.

        key:
            Dictionary key containing file path.

    Returns:
        Callable returning one index list
        per name in ``split_names``.
    """

    split_names = _normalize_names(split_names)

    def _inner(o, **kwargs):

        return tuple(
            _folder_idxs(
                o,
                name=name,
                folder="parent",
                key=key,
            )
            for name in split_names
        )

    _inner.split_names = split_names
    _inner.key = key

    return _inner


def FileSplitter(
    fname,
    split_names=("valid",),
    key="image",
    **kwargs,
):
    """
    Split items using filenames listed in a text file.

    The file should contain one filename per line.

    Supports multiple split names similarly to
    ``NameSplitter``.

    Expected format::

        train img1.jpg
        valid img2.jpg
        test img3.jpg

    Args:
        fname:
            Path to split-definition file.

        split_names:
            Split names to extract.

        key:
            Dictionary key containing file path.

    Returns:
        Callable returning one index list
        per split name.
    """

    split_names = _normalize_names(split_names)

    split_map = {}

    for line in Path(fname).read_text().splitlines():

        parts = line.strip().split()

        if len(parts) != 2:
            continue

        split_name, file_name = parts

        split_map.setdefault(split_name, set())
        split_map[split_name].add(file_name)

    def _inner(o, **kwargs):

        split_idxs = []

        for split_name in split_names:

            valid_files = split_map.get(split_name, set())

            idxs = [
                i for i, x in enumerate(o)
                if _get_path(x, key).name in valid_files
            ]

            split_idxs.append(idxs)

        return tuple(split_idxs)
    
    _inner.split_names = split_names
    _inner.key = key
    _inner.fname = fname

    return _inner


---

In [ ]:
#| export

@dataclass
class TargetedTransform:
    """
    Wrapper for a transform that specifies which input(s) it should be applied to.

    This allows fine-grained control when working with paired data such as
    (X, y), stereo images, or multi-modal inputs.

    Parameters
    ----------
    transform : callable
        The transform to apply. Must implement an `encodes()` method
        if used within a Transform pipeline.

    targets : tuple of str, default ("both",)
        Specifies where the transform should be applied.
        Supported values:
            - ("X",)      : apply only to the first element
            - ("y",)      : apply only to the second element
            - ("both",)   : apply to both elements

    Examples
    --------
    Apply to both inputs (default):

        TargetedTransform(RandomFlip())

    Apply only to X:

        TargetedTransform(RandomBrightness(), targets=("X",))

    Apply only to y:

        TargetedTransform(RemapMask(), targets=("y",))
    """
    transform: callable
    targets: tuple = ("both",)   # ("X",), ("y",), ("both",)


In [ ]:
#| export
def apply_transforms(image, transforms):
    """Apply a list of transformations, ensuring at least one is applied.
    
    Supports:
        - plain transforms (applied to both images if tuple)
        - TargetedTransform(transform, targets=...)
    """
    if not transforms:
        return image

    # Normalize transforms into TargetedTransform objects
    normalized = []
    for t in transforms:
        if isinstance(t, TargetedTransform):
            normalized.append(t)
        else:
            # Treat normal transforms as applied to both
            normalized.append(TargetedTransform(transform=t, targets=("both",)))

    # Randomly select transforms based on probability p if present
    applied = [
        spec for spec in normalized
        if not hasattr(spec.transform, "p") or rand() < spec.transform.p
    ]

    # Ensure at least one transform is applied
    if not applied:
        applied.append(choice(normalized))

    def apply_transform_to_image(img, transform):
        return transform.encodes(img)

    # ---- Single image case ----
    if not isinstance(image, tuple):
        for spec in applied:
            image = apply_transform_to_image(image, spec.transform)
        return image

    # ---- Tuple case ----
    image1, image2 = image

    for spec in applied:
        t = spec.transform
        targets = spec.targets

        if "both" in targets or "X" in targets:
            image1 = apply_transform_to_image(image1, t)

        if "both" in targets or "y" in targets:
            image2 = apply_transform_to_image(image2, t)

    return image1, image2

In [ ]:
# If we pass an empty list of transforms, it should return the input unchanged
test_eq(apply_transforms([1, 2], []), [1, 2]) 

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()